In [7]:
DECISIONS = []
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

import pandas as pd
import numpy as np

def remove_duplicates(df):
    df = df.copy()
    after = df.drop_duplicates()
    removed = len(df) - len(after)
    log('duplicates', 'dropped exact duplicate rows', removed)
    return after


def clean_unit_price(df, column ='unit_price'):
    before = df.copy()
    df = df.copy()
    df[column] = df[column].astype(str).str.replace('$', '', regex=False).str.strip()
    df[column] = pd.to_numeric(df[column], errors='coerce')
    log('clean unit prices', 'strip and to numeric', len(df.compare(before)))
    return df

def neg_quant(df, column = 'quantity'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df = df[df[column]>=0]
    removed = len(before) - len(df)
    log('negative quantity', 'drop negative quantities', removed)
    return df

def clean_store_id(df, column = 'store_id'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df[column] = df[column] = df[column].astype(str).str.strip().str.upper()
    log('clean store id', 'strip and upper', len(df.compare(before)))
    return df

def parse_timestamp(df, column = 'timestamp'):
    df = df.copy()
    rows_parsed = 0
    unparsed_count = 0
    if column in df.columns:
        values = df[column].astype('string').str.strip()
        parsed = pd.to_datetime(values, format='mixed', errors='coerce')
        unparsed_count = int((values.notna() & values.ne('') & parsed.isna()).sum())
        df[column] = parsed
        rows_parsed = int(parsed.notna().sum())
    log(
        'parse timestamps',
        f'parsed mixed formats; {unparsed_count} non-empty value(s) were unparseable',
        rows_parsed,
    )
    return df




DECISIONS = []

path = '../data/walmart_transactions.csv'
df = pd.read_csv(path)

df = remove_duplicates(df)
df = clean_unit_price(df)
df = neg_quant(df)
df = clean_store_id(df)
df = parse_timestamp(df)

def show_log():
    return pd.DataFrame(DECISIONS)

print(df.shape)

show_log()

[duplicates] dropped exact duplicate rows (30 row(s))
[clean unit prices] strip and to numeric (25532 row(s))
[negative quantity] drop negative quantities (130 row(s))
[clean store id] strip and upper (1785 row(s))
[parse timestamps] parsed mixed formats; 0 non-empty value(s) unparseable (24521 row(s))
(25402, 8)


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,clean unit prices,strip and to numeric,25532
2,negative quantity,drop negative quantities,130
3,clean store id,strip and upper,1785
4,parse timestamps,parsed mixed formats; 0 non-empty value(s) unp...,24521


In [8]:
df

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,FL-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82
...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,Diapers Size 3,baby,2,24.32
25558,TXN014835,FL-239,2024-09-26 08:47:08,CARDS-01,Playing Cards,Games & Entertainment,3,5.00
25559,TXN002434,FL-239,2024-09-16 16:39:00,PAPERTOWEL6,Paper Towels 6-Roll,household,2,7.78
25560,TXN014739,FL-710,2024-09-25 17:08:35,CHRG-PORT,PORTABLE CHARGER,electronics,1,24.18
